# Macroeconomic Data Exploratory Analysis

Module: Markets and Data

## Lesson summary

This notebook introduces exploratory analysis for macroeconomic series. It focuses on frequency, units, normalization, co-movement, and source limitations before students use the interactive macro dashboard.

## Learning objectives

By the end of this lesson, students should be able to:

- distinguish levels, rates, indexes, and exchange rates in macro data;
- normalize mixed-unit macro series for comparison;
- compute changes and rolling co-movement without confusing levels and returns;
- document publication frequency, release lag, and source limitations;
- prepare a macro panel for the Banxico/FRED dashboard.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src.fred import fred_series_catalog
from src.market_data import synthetic_macro_panel
from src.market_data_quality import data_quality_report, source_inventory_template

pd.set_option("display.max_columns", 80)

## Macro source inventory

In [ ]:
inventory = source_inventory_template()
inventory.loc[0] = {
    "provider": "Banxico SIE",
    "instrument_or_variable": "target rate, TIIE, CETES, FIX, UDI",
    "frequency": "daily, auction, or publication frequency",
    "start": "2020-01-01",
    "end": "2024-12-31",
    "field": "published value",
    "currency": "MXN, UDI, or percent",
    "calendar": "Mexican publication calendar",
    "known_limitations": "requires token; observations are not necessarily business-day complete",
}
inventory.loc[1] = {
    "provider": "FRED",
    "instrument_or_variable": "DGS10, DEXMXUS, MEXCPALTT01IXNBM",
    "frequency": "daily or monthly",
    "start": "2020-01-01",
    "end": "2024-12-31",
    "field": "value",
    "currency": "USD, MXN, index, or percent",
    "calendar": "US publication calendar",
    "known_limitations": "publication lag, revisions, and mixed frequencies",
}
inventory

In [ ]:
fred_series_catalog()

## Classroom macro panel

In [ ]:
macro = synthetic_macro_panel(periods=84)
macro.tail()

In [ ]:
data_quality_report(macro)

## Normalize mixed units

Rates, exchange rates, and index levels should not be plotted on the same axis without normalization.

In [ ]:
normalized = macro / macro.iloc[0] * 100
normalized.tail()

In [ ]:
ax = normalized.plot(figsize=(11, 5), title="Normalized macro panel")
ax.set_ylabel("Index = 100 at first observation")
plt.tight_layout()

## Changes and co-movement

In [ ]:
changes = macro[["banxico_target_rate", "mexico_inflation", "us_10y"]].diff()
returns = macro[["usd_mxn", "ipc_index"]].pct_change()

pd.concat(
    [
        changes.add_suffix("_change"),
        returns.add_suffix("_return"),
    ],
    axis=1,
).dropna().tail()

In [ ]:
rolling_corr = returns["ipc_index"].rolling(12).corr(returns["usd_mxn"])
ax = rolling_corr.plot(figsize=(10, 4), title="Rolling IPC versus USD/MXN co-movement")
ax.set_ylabel("12-period correlation")
plt.tight_layout()

## Macro interpretation checklist

| Question | Interpretation risk |
| --- | --- |
| Is the variable a level, rate, index, or return? | Transformations are not interchangeable. |
| Is the frequency daily, monthly, quarterly, or irregular? | Mixed frequencies can create artificial persistence. |
| Is the value revised after publication? | Historical analysis can use information unavailable at the time. |
| Which market calendar applies? | Mexican and US holidays do not line up perfectly. |
| Does the chart use real or classroom fallback data? | Claims from synthetic data should stay methodological, not empirical. |